In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

def plot_voxels_by_density_simple(
    mesh,
    ind_active,
    active_density,
    mode="abs",          # "abs" → use |density| for color/alpha, "raw" → use raw values
    cmap="viridis",
    alpha_min=0.05,
    alpha_max=0.95,
    edgecolor=None,
    linewidth=0.0,
    figsize=(7,6),
):
    """
    Minimal voxel renderer for SimPEG/discretize TensorMesh models.
    - Draws only active cells; opacity encodes density magnitude (or value).
    """
    # Expand active densities to full cell vector and reshape (discretize uses Fortran order)
    full = np.full(mesh.nC, np.nan)
    full[ind_active] = np.asarray(active_density, float)
    grid = full.reshape(mesh.shape_cells, order="F")

    # Build cell-edge grids once so physical sizes are correct
    x = mesh.x0[0] + np.r_[0.0, np.cumsum(mesh.h[0])]
    y = mesh.x0[1] + np.r_[0.0, np.cumsum(mesh.h[1])]
    z = mesh.x0[2] + np.r_[0.0, np.cumsum(mesh.h[2])]
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

    filled = ~np.isnan(grid)
    vals = np.abs(np.nan_to_num(grid)) if mode == "abs" else np.nan_to_num(grid)

    if np.any(filled):
        vmin = float(np.nanmin(vals[filled]))
        vmax = float(np.nanmax(vals[filled]))
        if not np.isfinite(vmin): vmin = 0.0
        if not np.isfinite(vmax) or vmax <= vmin: vmax = vmin + 1e-9
    else:
        vmin, vmax = 0.0, 1.0

    # Normalize → color + alpha
    t = np.clip((vals - vmin) / (vmax - vmin), 0.0, 1.0)
    rgba = cm.get_cmap(cmap)(t)
    rgba[..., 3] = alpha_min + (alpha_max - alpha_min) * t
    rgba[np.isnan(grid)] = 0.0  # fully transparent for inactive cells

    # Plot
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection="3d")
    ax.voxels(X, Y, Z, filled, facecolors=rgba, edgecolor=edgecolor, linewidth=linewidth)

    ax.set_box_aspect((np.sum(mesh.h[0]), np.sum(mesh.h[1]), np.sum(mesh.h[2])))
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    plt.tight_layout()
    plt.show()

plot_voxels_by_density_simple(
    mesh, ind_active, true_model,
    mode="abs", cmap="viridis",
    alpha_min=0.08, alpha_max=0.98,
    edgecolor=None, linewidth=0.0,
)

In [ ]:
import numpy as np
import pyvista as pv
from matplotlib import cm

def plot_voxels_pyvista_cells(
    mesh,
    ind_active,
    active_density,
    mode="abs",          # "abs": alpha from |density|; "raw": from signed value
    cmap="viridis",
    alpha_min=0.05,
    alpha_max=0.95,
    background="white",
    cpos="iso",
    show_edges=False,
):
    # 1) Full vector -> Fortran reshaped cube
    full = np.full(mesh.nC, np.nan, dtype=float)
    full[ind_active] = np.asarray(active_density, float)
    dens3 = full.reshape(mesh.shape_cells, order="F")
    vals = np.abs(np.nan_to_num(dens3)) if mode == "abs" else np.nan_to_num(dens3)

    # 2) Rectilinear grid from cell edges
    x = mesh.x0[0] + np.r_[0.0, np.cumsum(mesh.h[0])]
    y = mesh.x0[1] + np.r_[0.0, np.cumsum(mesh.h[1])]
    z = mesh.x0[2] + np.r_[0.0, np.cumsum(mesh.h[2])]
    grid = pv.RectilinearGrid(x, y, z)

    # 3) Attach cell scalars; mark actives
    grid.cell_data["density"] = dens3.ravel(order="F")
    active_ids = np.where(np.isfinite(grid.cell_data["density"]))[0]

    # 4) Keep only active cells as solid hexahedra (true voxels)
    solid = grid.cast_to_unstructured_grid().extract_cells(active_ids)

    # 5) Per-cell RGBA (alpha ~ density magnitude)
    d = solid.cell_data["density"]
    m = np.abs(np.nan_to_num(d)) if mode == "abs" else np.nan_to_num(d)
    if m.size and np.nanmax(m) > 0:
        t = (m - np.nanmin(m)) / max(1e-12, (np.nanmax(m) - np.nanmin(m)))
    else:
        t = np.zeros_like(m)

    # colormap → rgba in [0,1], then set alpha range
    rgba = cm.get_cmap(cmap)(t)
    rgba[:, 3] = alpha_min + (alpha_max - alpha_min) * t
    rgba_u8 = (rgba * 255).astype(np.uint8)
    solid.cell_data["rgba"] = rgba_u8  # per-cell colors with alpha

    # 6) Plot as voxels (cells), not a volume
    p = pv.Plotter()
    p.set_background(background)
    p.add_mesh(
        solid,
        scalars="rgba",
        rgba=True,                 # interpret scalars as RGBA
        show_edges=show_edges,
        ambient=1.0,               # flatter shading so colors/alpha read clearly
        lighting=False,            # optional: turn off lighting for clean look
    )
    p.add_axes(); p.show_grid(); p.camera_position = cpos
    p.show()

plot_voxels_pyvista_cells(mesh, ind_active, true_model)
